## Middleware

- These are basically hooks which can act as a trigger for a tool call or a function.
- Can be used to control what happens inside an agent.
- The hooks can be retries, fall backs, rate limits, tranforming prompts, tool selection and output formatting etc.


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

There are few in-built hooks too like 
- Summarization
- Human In Feedback
... more in Langchain

### Summarization
automatically summarize when approaching token limits

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage

agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:groq/compound-mini",  # We can use flash or mini models for summarization to save tokens
            trigger=("messages", 10),  # At every 10 messages summarization will work.
            keep=("messages", 4),  # To have enough context keep last 4 chats
        )
    ],
)

### What is a thread?

A thread is one conversation, identified by a `thread_id`. It is the key the `checkpointer` stores state under.

- Same `thread_id`: loads the saved messages, appends the new turn, saves again. This is what makes the agent remember.
- New `thread_id`: fresh conversation, empty state.
- No `config`: error, since the checkpointer has no key to save under.

The agent itself is stateless. Memory lives in the checkpointer, so one agent can serve many conversations (one `thread_id` per user).

`InMemorySaver` keeps state in RAM, so it is lost when the kernel restarts. Use `SqliteSaver` or `PostgresSaver` to persist.

In [3]:
config = {"configurable": {"thread_id": "testing-01"}}

In [4]:
import re
from IPython.display import Markdown, display


def clean(text):
    """qwen wraps its reasoning in <think> tags, which Markdown hides as HTML."""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()


def get_summary(messages):
    """The middleware injects the summary as a HumanMessage tagged lc_source."""
    return next(
        (
            m.content
            for m in messages
            if m.additional_kwargs.get("lc_source") == "summarization"
        ),
        None,
    )


def ask(agent, questions, config):
    last_summary = None
    for q in questions:
        messages = agent.invoke({"messages": [HumanMessage(content=q)]}, config)[
            "messages"
        ]

        display(Markdown(f"**Q:** {q}"))
        display(Markdown(clean(messages[-1].content)))

        summary = get_summary(messages)
        if summary and summary != last_summary:
            display(Markdown(f"**Summary injected:**\n\n{clean(summary)}"))
            last_summary = summary

        print(f"{len(messages)} messages in state\n")


questions_list = [
    "In one sentence, what is a vector database?",
    "Name two popular ones.",
    "Which one is easiest to run locally?",
    "How is it different from a normal SQL database?",
    "What is an embedding?",
    "Which model would you use to create embeddings?",
    "What does cosine similarity measure?",
    "Now, what was the very first thing I asked you about?",
]

ask(agent, questions_list, config)

**Q:** In one sentence, what is a vector database?

A vector database is a specialized database optimized for efficiently storing, indexing, and performing fast similarity searches on high-dimensional vector embeddings, primarily to power AI and machine learning applications.

2 messages in state



**Q:** Name two popular ones.

Two popular vector databases are **Pinecone** and **Milvus**.

...

Cosine similarity measures the **cosine of the angle between two vectors**, quantifying how similar their *directions* are while completely ignoring their *lengths* (magnitudes).

- **Range:** `-1` to `1` (most modern embedding models normalize vectors, so you'll typically see `0` to `1`)
- **`1.0`** → Vectors point in the exact same direction (maximum semantic similarity)
- **`0.0`** → Vectors are orthogonal (no meaningful relationship)
- **`-1.0`** → Vectors point in opposite directions (maximum dissimilarity)

**Why it matters for embeddings:**  
Two pieces of text with similar meanings will produce vectors that point in nearly the same direction, yielding a high cosine score—even if one is a short phrase and the other is a long paragraph. Because it focuses purely on orientation rather than scale, cosine similarity is the default metric for semantic search in virtually all vector databases.

8 messages in state



**Q:** Now, what was the very first thing I asked you about?

Based on the conversation history, the very first thing you asked about was **what a vector database is**.

10 messages in state



In [5]:
# What is stored in this thread right now. No API call.
for m in agent.get_state(config).values["messages"]:
    tag = m.additional_kwargs.get("lc_source", "")
    print(f"{type(m).__name__:12} {tag:14} {clean(m.content)[:70]!r}")

HumanMessage summarization  'Here is a summary of the conversation to date:\n\n## SESSION INTENT\nExpl'
AIMessage                   'Unlike a traditional SQL database that stores structured tabular data '
HumanMessage                'What is an embedding?'
AIMessage                   'An embedding is a dense numerical vector that converts unstructured da'
HumanMessage                'Which model would you use to create embeddings?'
AIMessage                   'The "best" embedding model depends on your **data type**, **performanc'
HumanMessage                'What does cosine similarity measure?'
AIMessage                   'Cosine similarity measures the **cosine of the angle between two vecto'
HumanMessage                'Now, what was the very first thing I asked you about?'
AIMessage                   'Based on the conversation history, the very first thing you asked abou'


In [6]:
# We can also trigger on tokens instead of message count.

token_agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:groq/compound-mini",
            trigger=("tokens", 500),  # Summarize once history passes 500 tokens.
            keep=("tokens", 200),  # Keep roughly the last 200 tokens.
        )
    ],
)

token_config = {"configurable": {"thread_id": "tokens-01"}}

ask(token_agent, questions_list[:5], token_config)

**Q:** In one sentence, what is a vector database?

A vector database is a specialized system designed to efficiently store, index, and retrieve high-dimensional vector embeddings based on semantic similarity rather than exact keyword matches.

2 messages in state



**Q:** Name two popular ones.

Two popular vector databases are **Pinecone** and **Qdrant**.

...

3 messages in state



**Q:** What is an embedding?

An **embedding** is a numerical representation of data (text, images, audio, etc.) that captures its meaning or key features as a fixed-length list of floating-point numbers. Instead of storing raw, unstructured data, AI models convert it into a vector where **semantically similar items end up close together** in a multi-dimensional space.

### 🔍 How it works (intuition + technical)
- **Input:** `"lightweight wireless earbuds for running"`
- **Model:** An embedding model (e.g., OpenAI `text-embedding-3-small`, `Sentence-BERT`, `CLIP`)
- **Output:** A vector like `[0.23, -0.87, 0.45, -0.12, ..., 0.31]` (typically 384–1536 dimensions)
- **Magic:** The model has learned during training that words/concepts with similar meanings produce similar vectors. So `"running headphones"` or `"sport earphones"` will yield vectors that sit nearby in that space.

### 📐 Key properties
| Property | Description |
|----------|-------------|
| **Dimensionality** | Fixed number of floats per vector (e.g., 768D). Higher isn't always better; it's a trade-off between accuracy and compute/storage cost. |
| **Similarity math** | Distance between vectors is measured using metrics like **cosine similarity**, **Euclidean distance**, or **dot product**. Closer = more similar. |
| **Dense & continuous** | Unlike sparse keyword indices (e.g., TF-IDF), embeddings capture nuanced, contextual meaning. |
| **Model-dependent** | Different models create different "semantic spaces." Vectors from Model A can't be directly compared to vectors from Model B. |

### 🤖 Why embeddings matter for AI & vector databases
- **Semantic search:** Find items by meaning, not exact keywords.
- **RAG (Retrieval-Augmented Generation):** Retrieve relevant documents before feeding them to an LLM.
- **Recommendations & clustering:** Group or match items based on hidden patterns.
- **Cross-modal AI:** Image embeddings can be matched with text embeddings (e.g., CLIP for "find images of a red bicycle").

### 🔗 Connection to your previous context
A vector database like **Qdrant** doesn't create embeddings; it **stores and indexes them** so you can run fast Approximate Nearest Neighbor (ANN) searches. The SQL database handles structured metadata (IDs, prices, categories), while the vector DB handles the semantic matching layer. Together, they power modern AI applications.

Want a quick code snippet showing how to generate an embedding and store it in Qdrant?

**Summary injected:**

Here is a summary of the conversation to date:

## SESSION INTENT
Explain how a vector database (e.g., Qdrant) differs from a traditional SQL database.

## SUMMARY
- **Qdrant** is an open‑source, self‑hosted vector database that can be run locally (e.g., `docker run -p 6333:6333 qdrant/qdrant`). It stores high‑dimensional embeddings and provides Approximate Nearest Neighbor (ANN) search via indexes such as HNSW, IVF, PQ.
- **SQL databases** store structured data in tables with rows, columns, and strict schemas. They use B‑tree, hash, GiST/GIN indexes and support exact‑match queries, joins, aggregations, and ACID transactions.
- **Key differences**  

| Aspect | SQL Database | Vector Database (Qdrant) |
|--------|--------------|--------------------------|
| Data Model | Tables, rows, columns, strong schema | Collections of vectors (embeddings) + flexible metadata |
| Query Paradigm | Exact matches, `SELECT`/`WHERE`/`JOIN`, aggregations | ANN similarity search (`find items most similar to this vector`) |
| Indexing | B‑trees, hash, GiST/GIN (optimized for exact/range lookups) | HNSW, IVF, PQ, LSH (optimized for high‑dimensional similarity) |
| Core Use Cases | Transactions, reporting, CRUD, relational data | Semantic search, recommendation, RAG, clustering, anomaly detection |
| Performance Focus | Fast deterministic reads/writes, ACID compliance | High‑throughput similarity calculations, approximate results |
| Schema Flexibility | Strongly typed, schema‑enforced | Schema‑less or loosely typed; vectors + optional JSON metadata |

- **Practical example**  
  - SQL: `SELECT title, price FROM products WHERE category='Headphones' AND price < 100;` (exact filters)  
  - Vector DB: “Show me products similar to ‘lightweight wireless earbuds for running’” → embed query, retrieve nearest vectors.

- **Complementary nature**: AI applications often use both; SQL for structured metadata/transactions, vector DB for semantic retrieval. Hybrid solutions exist (e.g., pgvector, MongoDB Atlas Vector Search) where IDs in the vector store reference rows in a SQL store.

## ARTIFACTS
None

## NEXT STEPS
Await further user queries (e.g., request for integration example, performance tuning, or other comparisons).

3 messages in state



In [7]:
# Or trigger on a fraction of the model's context window, which adapts to the model.

fraction_agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:groq/compound-mini",
            trigger=("fraction", 0.005),  # Summarize at 0.5% of the context window.
            keep=("fraction", 0.002),  # Keep the last 0.2%.
        )
    ],
)

fraction_config = {"configurable": {"thread_id": "fraction-01"}}

ask(fraction_agent, questions_list[:4], fraction_config)

**Q:** In one sentence, what is a vector database?

A vector database is a specialized storage system that indexes and retrieves high-dimensional numerical representations (embeddings) of data to enable fast, semantically meaningful similarity searches.

2 messages in state



**Q:** Name two popular ones.

Two popular vector databases are **Pinecone** and **Milvus**.

...

3 messages in state



**Q:** How is it different from a normal SQL database?

Vector databases like **Milvus** and traditional **SQL (relational) databases** solve fundamentally different problems. Here’s how they compare across key dimensions:

| Aspect | SQL Database | Vector Database (e.g., Milvus) |
|--------|--------------|--------------------------------|
| **Data Format** | Structured tables: rows, columns, schemas | High-dimensional numerical arrays (embeddings) representing meaning/features |
| **Primary Purpose** | Store & manage relational data, enforce business rules, run transactions | Store & search embeddings for similarity, semantic understanding, or ML features |
| **Query Types** | `SELECT`, `JOIN`, `WHERE`, aggregations, exact/pattern matches | Approximate Nearest Neighbor (ANN), cosine similarity, Euclidean distance, hybrid filters |
| **Indexing** | B-trees, hash indexes, full-text indexes | HNSW, IVF, IVF-PQ, DiskANN (optimized for multi-dimensional space traversal) |
| **Accuracy vs Speed** | Exact matches; deterministic results | Trades slight accuracy for massive speed gains at scale (ANN vs exact KNN) |
| **Typical Use Cases** | User accounts, orders, inventory, financial records, reporting | Semantic search, recommendation engines, RAG/chatbots, image/audio similarity, anomaly detection |

### Key Conceptual Differences
1. **How "closeness" is defined**  
   - SQL: `WHERE status = 'active' AND age > 25` (exact or range-based)  
   - Vector DB: `"Find embeddings within 0.1 cosine distance of this query vector"` (geometric proximity in 512–4096+ dimensions)

2. **Indexing Mechanics**  
   SQL indexes sort data linearly or hierarchically for fast lookups. Vector indexes build graph or clustering structures that let the engine skip most of the dataset while still finding semantically similar vectors. This is necessary because comparing every vector in a large collection is computationally prohibitive.

3. **Schema & Flexibility**  
   SQL enforces strict schemas and relationships. Vector databases are schema-light; you store vectors alongside optional metadata (e.g., document IDs, timestamps, tags) and filter on that metadata during similarity search.

### They’re Complementary, Not Competitive
- Most production AI applications use **both**: SQL for user data, permissions, and transactions; vector DBs for retrieval/embedding search.
- The line is blurring: PostgreSQL (`pgvector`), MySQL, and SQLite now support vector indexing, while Milvus supports vector + scalar filtering. But purpose-built vector DBs like Milvus still win on scale, specialized ANN algorithms, and distributed search performance.

**Bottom line:** Use SQL when you need exact matches, relationships, and ACID transactions. Use a vector database when you need to find things that are *similar in meaning or feature space*, especially at scale with AI/ML workloads.

**Summary injected:**

Here is a summary of the conversation to date:

## SESSION INTENT
Determine which of the two previously mentioned vector databases (Pinecone, Milvus) is easiest to run locally.

## SUMMARY
- **Pinecone**: Fully managed cloud‑only service; no self‑hosted or local deployment option.
- **Milvus**: Open‑source and designed for local/self‑hosted use. Provides straightforward local deployment options:
  - *Milvus Lite*: Lightweight single‑binary version for local development (in‑memory or simple file storage, zero extra infrastructure).
  - *Docker Compose*: Official `docker-compose.yml` spins up a full Milvus cluster (Milvus + etcd + MinIO + message queue) with a single command.
- Mentioned additional ultra‑lightweight local alternatives (e.g., Chroma, LanceDB) for context, but between Pinecone and Milvus, **Milvus** is the only viable local option.

## ARTIFACTS
None.

## NEXT STEPS
None; the user's question has been fully answered.

3 messages in state



## Human In Loop Middleware

Pause agent execution for human approval, editing or rejection.
Usecase:
- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [8]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command


def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Send an email. Mock implementation."""
    return f"Email sent to {recipient} successfully."


def read_email_tool(email_id: str) -> str:
    """Read an email by id. Mock implementation."""
    return "Email content: [content-here]"

In [9]:
hitl_agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    tools=[send_email_tool, read_email_tool],  # Tools must be registered to be called.
    checkpointer=InMemorySaver(),  # Required, since an interrupt has to be resumed later.
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {"allowed_decisions": ["approve", "edit", "reject"]},
                "read_email_tool": False,  # False means run it without asking.
            }
        )
    ],
)

In [10]:
# Step 1: the agent pauses instead of sending. Note the __interrupt__ key.

approve_config = {"configurable": {"thread_id": "test-approve"}}

result = hitl_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send an email to john@test.com with subject 'Hello' and body 'How are you?'"
            )
        ]
    },
    approve_config,
)

request = result["__interrupt__"][0].value
for action in request["action_requests"]:
    print(action["name"], action["args"])

send_email_tool {'body': 'How are you?', 'recipient': 'john@test.com', 'subject': 'Hello'}


In [11]:
# Step 2: approve. The tool runs and the agent continues from where it paused.

result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    approve_config,
)
print(clean(result["messages"][-1].content))

The email has been sent to john@test.com successfully.


In [12]:
# Editing: change the tool args before the tool runs.

edit_config = {"configurable": {"thread_id": "test-edit"}}

result = hitl_agent.invoke(
    {"messages": [HumanMessage(content="Send an email to john@test.com saying hello")]},
    edit_config,
)

action = result["__interrupt__"][0].value["action_requests"][0]
print("proposed:", action["args"])

edited_args = {**action["args"], "recipient": "jane@test.com"}

result = hitl_agent.invoke(
    Command(
        resume={
            "decisions": [
                {"type": "edit", "edited_action": {"name": action["name"], "args": edited_args}}
            ]
        }
    ),
    edit_config,
)

for m in result["messages"]:
    if m.type == "tool":
        print("tool ran with:", m.content)

proposed: {'body': 'Hello!', 'recipient': 'john@test.com', 'subject': 'Hello'}


tool ran with: Email sent to jane@test.com successfully.


In [13]:
# Rejecting instead. The message tells the model why, so it can adapt.

reject_config = {"configurable": {"thread_id": "test-reject"}}

hitl_agent.invoke(
    {
        "messages": [
            HumanMessage(content="Send an email to spam@test.com saying 'buy now'")
        ]
    },
    reject_config,
)
result = hitl_agent.invoke(
    Command(
        resume={
            "decisions": [{"type": "reject", "message": "Do not send marketing email."}]
        }
    ),
    reject_config,
)
print(clean(result["messages"][-1].content))

I cannot send that email because the system identified it as marketing content. I am restricted from sending emails that appear to be spam or promotional in nature.


In [14]:
# read_email_tool was set to False, so it runs with no interrupt at all.

read_config = {"configurable": {"thread_id": "test-read"}}

result = hitl_agent.invoke(
    {"messages": [HumanMessage(content="Read email with id abc-123")]},
    read_config,
)
print("interrupted:", "__interrupt__" in result)
print(clean(result["messages"][-1].content))

interrupted: False
The email with id `abc-123` contains the following content:

[content-here]


In [15]:
# Editing

edit_config = {"configurable": {"thread_id": "test-edit"}}

result = hitl_agent.invoke(
    {
        "message": (
            HumanMessage(
                content="Sned a email to wong@email with subject 'new born'  and body 'it's a boy' "
            )
        )
    },
    edit_config,
)